#  Machine Learning Fundamentals



---

## Learning Objectives

- Explain supervised, unsupervised, and reinforcement learning.
- Describe the end-to-end ML pipeline.
- Load and explore a real dataset with Pandas.
- Split data into training and test sets.
- Scale features with StandardScaler.
- Train a Logistic Regression classifier.
- Evaluate a model using accuracy, confusion matrix, and classification report.

## Concept / Theory

### What is Machine Learning?
ML is the field where algorithms learn patterns from data to make predictions or decisions — without being explicitly programmed for each task.

### Three Types of ML

| Type | Input | Goal | Example |
|---|---|---|---|
| **Supervised** | Data + Labels | Learn input→output mapping | Email spam detection |
| **Unsupervised** | Data only | Discover structure | Customer segmentation |
| **Reinforcement** | Environment rewards | Learn optimal actions | Game-playing AI |

### The ML Pipeline
```
Data Collection → Data Cleaning → Feature Engineering
       ↓
  Train/Test Split → Scale Features
       ↓
  Train Model → Evaluate → Tune → Deploy
```

### Key Concepts
- **Features (X):** input variables (columns)
- **Target (y):** the value to predict (label)
- **Training set:** used to fit model parameters
- **Test set:** used to measure real-world performance
- **Overfitting:** model memorizes training data, fails on new data
- **Underfitting:** model too simple, fails even on training data

In [ ]:
# Lab 1: Setup & imports
!pip install -q scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

print('All libraries loaded successfully!')

## Practical Lab 1 – Setup & the ML Workflow

### Objective
Understand the full ML workflow on a simple toy example.

### Explanation
Before touching a real dataset, we trace through the complete ML pipeline step-by-step.

In [ ]:
# Lab 1: Toy ML workflow from scratch
np.random.seed(42)

# Step 1: Data — study_hours -> pass/fail
X_toy = np.array([[1],[2],[3],[4],[5],[6],[7],[8],[9],[10]])
y_toy = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1])

print('Feature matrix X (study hours):', X_toy.ravel())
print('Labels y (0=Fail, 1=Pass):', y_toy)

# Step 2: Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(X_toy, y_toy, test_size=0.2, random_state=42)
print(f'\nTrain size: {len(X_tr)} | Test size: {len(X_te)}')

# Step 3: Scale
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

# Step 4: Train
model = LogisticRegression()
model.fit(X_tr_sc, y_tr)

# Step 5: Predict & evaluate
preds = model.predict(X_te_sc)
print('Predictions:', preds)
print('Actual:     ', y_te)
print('Accuracy:', accuracy_score(y_te, preds))

## Practical Lab 2 – Exploring the Iris Dataset

### Objective
Load, profile, and visualize the classic Iris dataset.

### Explanation
The Iris dataset contains measurements of 150 iris flowers from 3 species. It is the "Hello World" of classification.

In [ ]:
# Lab 2: Load Iris dataset
iris = datasets.load_iris()
X    = iris.data
y    = iris.target

# Create DataFrame for exploration
df = pd.DataFrame(X, columns=iris.feature_names)
df['target']  = y
df['species'] = df['target'].map(dict(enumerate(iris.target_names)))

print('Shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nClass distribution:')
print(df['species'].value_counts())

In [ ]:
# Lab 2: Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pairplot substitute — scatter
colors = ['red','green','blue']
for i, (species, color) in enumerate(zip(iris.target_names, colors)):
    mask = y == i
    axes[0].scatter(X[mask, 0], X[mask, 1], c=color, label=species, alpha=0.7)
axes[0].set_xlabel('Sepal length (cm)')
axes[0].set_ylabel('Sepal width (cm)')
axes[0].set_title('Sepal dimensions')
axes[0].legend()

# Feature distributions
for i, col in enumerate(iris.feature_names):
    axes[1].hist(X[:, i], alpha=0.5, label=col.split('(')[0], bins=15)
axes[1].set_title('Feature distributions')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

print('\nStatistical summary:')
print(df.describe().round(2))

### Expected Output
A scatter plot of sepal dimensions with 3 colored clusters and a histogram of feature distributions.

## Practical Lab 3 – Train/Test Split & Feature Scaling

### Objective
Split the dataset properly and apply StandardScaler.

### Explanation
- **Train/test split** prevents data leakage — the model must never see test data during training.
- **Stratification** ensures each class is represented equally in both splits.
- **StandardScaler** transforms each feature to have mean=0 and std=1, which helps gradient-based models converge.

In [ ]:
# Lab 3: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for testing
    random_state=42,      # reproducible
    stratify=y            # balanced classes
)
print(f'Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test samples:     {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)')

# Check class balance
for split, labels in [('Train', y_train), ('Test', y_test)]:
    counts = np.bincount(labels)
    print(f'{split} class counts: {dict(enumerate(counts))}')

In [ ]:
# Lab 3: Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform
X_test_sc  = scaler.transform(X_test)        # transform only!

print('Before scaling (train):')
print('  Mean:', X_train[:, 0].mean().round(3), '  Std:', X_train[:, 0].std().round(3))

print('After scaling (train):')
print('  Mean:', X_train_sc[:, 0].mean().round(6), '  Std:', X_train_sc[:, 0].std().round(3))

print('\n⚠️ Key rule: Always fit scaler on TRAINING data only!')
print('   Then transform both train and test with the SAME fitted scaler.')

### Expected Output
```
After scaling — Mean: ~0.0  Std: ~1.0
```

## Practical Lab 4 – First Classifier: Logistic Regression

### Objective
Train a Logistic Regression model and make predictions.

### Explanation
Logistic Regression is a linear classifier that outputs probabilities. Despite its name, it is used for **classification**. It learns a decision boundary using the sigmoid function.

In [ ]:
# Lab 4: Train Logistic Regression
clf = LogisticRegression(max_iter=200, random_state=42)
clf.fit(X_train_sc, y_train)

print('Model trained!')
print('Classes:', clf.classes_)
print('Coefficients shape:', clf.coef_.shape)

# Predict on test set
y_pred = clf.predict(X_test_sc)
y_prob = clf.predict_proba(X_test_sc)

print('\nFirst 10 predictions:',  y_pred[:10])
print('Actual labels:       ',  y_test[:10])
print('\nSample probabilities (first 3 rows):')
for i, (prob, pred, actual) in enumerate(zip(y_prob[:3], y_pred[:3], y_test[:3])):
    print(f'  Row {i}: [{prob[0]:.2f}, {prob[1]:.2f}, {prob[2]:.2f}] => pred={iris.target_names[pred]}, actual={iris.target_names[actual]}')

## Practical Lab 5 – Model Evaluation Metrics

### Objective
Understand accuracy, confusion matrix, and classification report.

### Explanation
- **Accuracy** = correct predictions / total predictions
- **Confusion matrix** shows true positives, false positives, etc.
- **Precision** = TP / (TP + FP) — of predicted positives, how many are correct?
- **Recall** = TP / (TP + FN) — of actual positives, how many did we catch?
- **F1-Score** = harmonic mean of precision and recall

In [ ]:
# Lab 5: Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f} ({acc*100:.1f}%)')

# Classification report
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Lab 5: Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Confusion Matrix — Logistic Regression on Iris')
plt.tight_layout()
plt.show()

print('\nConfusion Matrix (raw):')
print(pd.DataFrame(cm, index=iris.target_names, columns=iris.target_names))

### Expected Output
Accuracy ~97%. The confusion matrix shows most predictions on the diagonal (correct). Setosa is typically 100% accurate.

## Exercises

**Exercise 1:** Using the Iris dataset, change the `test_size` to 0.3 and compare accuracy.

In [ ]:
# Exercise 1
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
# Change test_size and compare accuracy

**Exercise 2:** Try Logistic Regression without scaling. Compare accuracy with and without scaling.

In [ ]:
# Exercise 2
# Train without scaling and compare accuracy
# clf_unscaled = LogisticRegression(max_iter=200, random_state=42)
# ...

**Exercise 3:** Print the feature importances (coefficients) to find which feature matters most.

In [ ]:
# Exercise 3
# clf.coef_ has shape (n_classes, n_features)
# Print a DataFrame of feature names and their coefficients

## Challenge Task

### Predict Wine Quality
Load Scikit-learn's `load_wine()` dataset. Run the full pipeline:
1. Load & profile
2. Train/test split (80/20, stratified)
3. Scale with StandardScaler
4. Train LogisticRegression
5. Print accuracy and classification report
6. Visualize confusion matrix

In [ ]:
# Challenge Task: Wine Classification
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import pandas as pd, matplotlib.pyplot as plt

wine = load_wine()
X, y = wine.data, wine.target

print(f'Samples: {X.shape[0]} | Features: {X.shape[1]} | Classes: {len(wine.target_names)}')
print('Class names:', wine.target_names)

# Your full pipeline here
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)
clf = LogisticRegression(max_iter=300, random_state=42)
clf.fit(X_tr, y_train)
y_pred = clf.predict(X_te)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=wine.target_names))

## Real-World AI/ML Application

### Where is Logistic Regression Used?

| Application | Prediction |
|---|---|
| Email filtering | Spam vs. Not Spam |
| Medical screening | Disease vs. No disease |
| Credit scoring | Default vs. No default |
| Document classification | Category label |

Logistic Regression is the **baseline** model you should always try first before more complex algorithms. It is fast, interpretable, and often surprisingly accurate.

## Day Summary

| Concept | Key Takeaway |
|---|---|
| Supervised ML | Learn from labeled data (X → y) |
| ML Pipeline | Collect → Clean → Split → Scale → Train → Evaluate |
| Train/test split | Never evaluate on training data |
| StandardScaler | Normalize features to mean=0, std=1 |
| Logistic Regression | Linear classifier → probabilities |
| Accuracy | Fraction of correct predictions |
| Confusion matrix | True/false positives/negatives |
| F1-Score | Balance of precision & recall |

